# Detecção de anomalias em transações Bitcoin

Pipeline de processamento em lote sobre arquitetura medalhão no BigQuery, com
K-Means treinado em BigQuery ML para identificar transações estruturalmente
atípicas. Recorte de 2020: **112.553.498 transações** em 53.222 blocos.

Todo o processamento e toda a modelagem estão escritos em SQL. Este notebook não
contém SQL nenhum: ele lê os 41 arquivos do repositório, resolve os marcadores de
ambiente e submete cada um ao BigQuery. Os arquivos continuam sendo a fonte única
de verdade.

## Como as etapas estão divididas aqui

O pipeline completo leva dezenas de minutos e move o ano inteiro entre o BigQuery
e o Cloud Storage. Não cabe em cinco minutos. Em vez de escolher entre não mostrar
as etapas pesadas e arriscar executá-las ao vivo, elas aparecem em **três camadas
de profundidade**, cada uma provando uma coisa diferente:

| Camada | O que faz | Cobertura | O que prova |
|---|---|---|---|
| **Verificar** (§4) | *dry run*, nada executa | as 27 etapas de escrita | o SQL é válido, e quanto custaria |
| **Demonstrar** (§5) | executa de verdade, recorte de **um dia**, destino descartável | `20`, `25`, `34` | o SQL roda sobre dados reais |
| **Apresentar** (§6) | executa de verdade, **ano completo** | as 14 etapas de leitura | o resultado |

O que torna a camada do meio possível é o **particionamento por data**: como toda
tabela pesada é particionada por dia, trocar o recorte de 366 dias por 1 corta a
leitura na mesma proporção, sem alterar uma linha da lógica.

## 1. Ambiente

> **A conta de faturamento é emprestada.** Por isso a primeira coisa configurada
> aqui é um **teto de bytes faturados**. Se uma consulta fosse exceder o teto, o
> BigQuery a recusa *antes* de processar — a consulta falha de graça, em vez de
> gerar uma fatura para outra pessoa. É a mesma proteção que
> `require_partition_filter` dá no nível da tabela, aplicada no nível do job.

In [ ]:
# Em BigQuery Studio (Colab Enterprise) tudo isto ja vem instalado.
# Localmente:  pip install google-cloud-bigquery pandas matplotlib db-dtypes
import sys, pathlib, re, time, difflib

sys.path.insert(0, str(pathlib.Path.cwd()))
import pipeline_lib as pl
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

pl.aplicar_estilo()
P = pl.PALETA

cfg = pl.carregar_config()
etapas = pl.descobrir_etapas()
print(f"{len(etapas)} etapas | projeto {cfg['PROJECT_ID']} | local {cfg['LOCATION']}")

In [ ]:
from google.cloud import bigquery

cliente = bigquery.Client(project=cfg["PROJECT_ID"], location=cfg["LOCATION"])

# Teto por job. Nenhuma consulta deste notebook chega perto disto: a mais pesada
# e a 37, que cruza anomaly_scores com tx_enriched no ano inteiro.
TETO_BYTES = 50 * 1024**3   # 50 GB

def config_job(**extra):
    return bigquery.QueryJobConfig(maximum_bytes_billed=TETO_BYTES, **extra)

print(f"cliente pronto | teto por consulta: {pl.humanizar_bytes(TETO_BYTES)}")

### As três funções de execução

Cada camada tem a sua, e **nenhuma consegue escrever em produção**. A guarda falha
fechada: qualquer instrução de escrita bloqueia, e uma escrita cujo alvo não foi
identificado também bloqueia. `EXPORT DATA` é o caso que justifica esse rigor —
ele grava no Cloud Storage, não num dataset, então passaria despercebido por
qualquer checagem baseada em nome de dataset.

In [ ]:
DIA_DEMO     = "2020-06-01"   # a particao que a camada "Demonstrar" processa
DATASET_DEMO = "demo"         # destino descartavel; nunca bronze/silver/gold


def verificar(etapa):
    """Camada 1. Dry run: valida a sintaxe e mede o custo sem executar nada."""
    sql = pl.resolver(etapa.sql_bruto(), cfg)
    job = cliente.query(sql, job_config=bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False))
    return job.total_bytes_processed


def consultar(etapa):
    """Camada 3. Executa uma etapa de leitura sobre o ano completo."""
    sql = pl.resolver(etapa.sql_bruto(), cfg)
    pl.exigir_somente_leitura(sql)
    faltando = sorted(pl.objetos_exigidos(sql) - PRESENTES)
    if faltando:
        print(f"[{etapa.numero:02d}] indisponivel. Falta materializar: "
              + ", ".join(faltando))
        return None
    return cliente.query(sql, job_config=config_job()).result().to_dataframe()


def demonstrar(etapa, dia=None):
    """Camada 2. Executa a etapa de verdade sobre um dia, escrevendo no demo."""
    sql = pl.reescrever_para_demo(
        pl.resolver(etapa.sql_bruto(), cfg), dia or DIA_DEMO, DATASET_DEMO)
    pl.exigir_somente_leitura(sql, permitidos=[DATASET_DEMO])
    t0 = time.time()
    job = cliente.query(sql, job_config=config_job())
    job.result()
    return sql, time.time() - t0, job


ds = bigquery.Dataset(f"{cfg['PROJECT_ID']}.{DATASET_DEMO}")
ds.location = cfg["LOCATION"]
ds.description = "Saidas da demonstracao ao vivo. Descartavel."
cliente.create_dataset(ds, exists_ok=True)
print(f"dataset de demonstracao pronto: {DATASET_DEMO} | dia: {DIA_DEMO}")

## 2. O pipeline

As cinco pastas do repositório são as cinco etapas do trajeto, e a numeração é
global e contínua de `01` a `41`: ela codifica a ordem de execução, e as
dependências entre os arquivos são rígidas.

O inventário abaixo consulta a **API de metadados**, não uma query SQL — lista
tabelas e modelos sem processar um único byte, então não consome nada do teto nem
da conta emprestada. Ele confirma que o pipeline foi executado ponta a ponta.

In [ ]:
DEMO = [20, 25, 34]   # as tres etapas da camada "Demonstrar"

def camada_apresentacao(e):
    if e.numero in DEMO:
        return "Demonstrar"
    return "Verificar" if e.escreve else "Apresentar"

PRESENTES = pl.inventario(cliente, cfg)

print(f"{len(PRESENTES)} objetos materializados\n")
for dsn in pl.DATASETS_PRODUCAO:
    nomes = sorted(o.split(".", 1)[1] for o in PRESENTES if o.startswith(dsn + "."))
    print(f"  {dsn:7s} {', '.join(nomes) if nomes else '(vazio)'}")

diag = pd.DataFrame(pl.diagnostico(etapas, PRESENTES, cfg)).set_index("numero")
diag["camada_apres"] = [camada_apresentacao(e) for e in etapas]
bloqueadas = diag[~diag["pode_rodar"]]

if len(bloqueadas):
    print(f"\n{len(bloqueadas)} etapa(s) com dependencia ausente:")
    display(bloqueadas[["camada", "arquivo", "faltando"]])
else:
    print(f"\nAs {len(diag)} etapas tem todas as dependencias satisfeitas: "
          "o pipeline esta completo.")

In [ ]:
diag[["camada", "arquivo", "verbo", "camada_apres"]]

## 3. O que já rodou

A evidência de que o caminho pesado foi percorrido. Os números vêm do que o
servidor cobrou, não de um cronômetro do cliente.

A fonte preferencial é o manifesto gravado por `00-run-pipeline.ipynb`. Na
ausência dele — porque as queries foram executadas direto no console — o notebook
cai para `INFORMATION_SCHEMA.JOBS_BY_PROJECT`, que registra todo job do projeto
independentemente de quem o disparou.

In [ ]:
DIAS_ATRAS       = 30   # janela do fallback
SEGUNDOS_MINIMOS = 2    # descarta jobs triviais no fallback


def linha_do_tempo():
    man = pl.ler_manifesto()
    if man and any(e["status"] == "ok" for e in man["etapas"]):
        df = pd.DataFrame([e for e in man["etapas"]
                           if e["status"] == "ok" and e["inicio"]])
        df["inicio_ts"] = pd.to_datetime(df["inicio"], format="mixed", utc=True)
        df["rotulo"] = df.apply(
            lambda r: f"{r['numero']:02d} {r['arquivo'][3:-4]}", axis=1)
        return df, "manifesto de 00-run-pipeline.ipynb"

    regiao = "region-" + cfg["LOCATION"].lower()
    sql = f"""
    SELECT job_id, start_time, end_time, statement_type, query,
           TIMESTAMP_DIFF(end_time, start_time, MILLISECOND) / 1000 AS segundos,
           total_bytes_billed, total_slot_ms
    FROM `{regiao}`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
    WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL @dias DAY)
      AND job_type = 'QUERY' AND state = 'DONE'
      AND error_result IS NULL AND parent_job_id IS NULL
    ORDER BY start_time
    """
    jc = config_job(query_parameters=[
        bigquery.ScalarQueryParameter("dias", "INT64", DIAS_ATRAS)])
    df = cliente.query(sql, job_config=jc).result().to_dataframe()
    df = df[df["segundos"] >= SEGUNDOS_MINIMOS].copy()
    df["rotulo"] = df["query"].map(
        lambda q: (re.search(r"`([a-z0-9_]+\.[a-z0-9_]+)`", q or "",
                             re.IGNORECASE) or [None, "(sem alvo)"])[1]
        if re.search(r"`([a-z0-9_]+\.[a-z0-9_]+)`", q or "", re.IGNORECASE)
        else "(sem alvo)")
    df["camada"] = df["rotulo"].str.split(".").str[0]
    df["bytes_faturados"] = df["total_bytes_billed"]
    df["inicio_ts"] = pd.to_datetime(df["start_time"], utc=True)
    return df, f"INFORMATION_SCHEMA, ultimos {DIAS_ATRAS} dias"


tempos, origem = linha_do_tempo()
print(f"{len(tempos)} jobs | origem: {origem}")

In [ ]:
# Gantt do pipeline. Uma barra por job, posicionada no tempo real de inicio.
df = tempos.sort_values("inicio_ts").reset_index(drop=True)
df["offset"] = (df["inicio_ts"] - df["inicio_ts"].min()).dt.total_seconds()

fig, ax = plt.subplots(figsize=(10, max(3.5, 0.26 * len(df))))
for y, r in df.iterrows():
    ax.barh(y, r["segundos"], left=r["offset"], height=0.6,
            color=pl.COR_CAMADA.get(r.get("camada"), P["serie"][0]))

ax.set_yticks(range(len(df)))
ax.set_yticklabels(df["rotulo"], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("segundos desde o inicio do pipeline")
ax.set_title("Execucao real do pipeline", loc="left")
ax.grid(axis="y", visible=False)
ax.set_axisbelow(True)

vistas = [c for c in pl.COR_CAMADA if c in set(df.get("camada", []))]
if vistas:
    ax.legend(handles=[Patch(facecolor=pl.COR_CAMADA[c], label=c) for c in vistas],
              loc="lower right", bbox_to_anchor=(1, 1.01), ncol=len(vistas))

plt.tight_layout()
plt.show()

print(f"tempo somado: {pl.humanizar_segundos(df['segundos'].sum())} | "
      f"faturado: {pl.humanizar_bytes(df.get('bytes_faturados', pd.Series(dtype=float)).sum())}")

## 4. Camada 1 — Verificar

*Dry run* das 27 etapas de escrita. O BigQuery analisa cada query, valida a
sintaxe e as referências, e devolve quantos bytes ela leria — **sem executar nada
e sem custo**. É a prova ao vivo de que o SQL do repositório está íntegro, e o
número da direita é o que a execução real custaria.

In [ ]:
escritas = [e for e in etapas if e.escreve]
linhas, falhas = [], []

for e in escritas:
    try:
        linhas.append({"n": e.numero, "camada": e.camada,
                       "arquivo": e.arquivo, "bytes": verificar(e)})
    except Exception as erro:
        falhas.append({"n": e.numero, "arquivo": e.arquivo, "erro": str(erro)[:160]})

verificacao = pd.DataFrame(linhas)
verificacao["leria"] = verificacao["bytes"].map(pl.humanizar_bytes)

print(f"{len(linhas)} de {len(escritas)} etapas validaram sintaxe e referencias")
for f in falhas:
    print(f"  {f['n']:02d} {f['arquivo']}: {f['erro']}")
print(f"\nleitura total se o pipeline rodasse de novo agora: "
      f"{pl.humanizar_bytes(verificacao['bytes'].sum())}")

verificacao[["n", "camada", "arquivo", "leria"]].set_index("n")

## 5. Camada 2 — Demonstrar

As três etapas do miolo do pipeline — enriquecimento, matriz de features e
detecção — executadas **de verdade, agora**, sobre a partição de um único dia.
Elas leem as tabelas de produção do ano completo e escrevem no dataset `demo`.

A célula abaixo mostra as duas únicas diferenças em relação ao arquivo do
repositório, para conferir a olho que a lógica não foi tocada.

In [ ]:
alvo = pl.por_numero(etapas, 25)
original = pl.resolver(alvo.sql_bruto(), cfg)
reescrito = pl.reescrever_para_demo(original, DIA_DEMO, DATASET_DEMO)

print("\n".join(difflib.unified_diff(
    original.splitlines(), reescrito.splitlines(),
    "repositorio", "demonstracao", lineterm="", n=0)))

In [ ]:
# A camada 2 PROCESSA dados. Deixe em False para apresentar sem consumir cota.
EXECUTAR_DEMO = True

resultados = []
for numero in (DEMO if EXECUTAR_DEMO else []):
    e = pl.por_numero(etapas, numero)
    print(f"[{numero:02d}] {e.arquivo} ... ", end="", flush=True)
    sql, segundos, job = demonstrar(e)
    destino = re.search(r"CREATE OR REPLACE TABLE `([^`]+)`", sql).group(1)
    n_linhas = cliente.get_table(f"{cfg['PROJECT_ID']}.{destino}").num_rows
    resultados.append({"n": numero, "etapa": e.objeto, "destino": destino,
                       "segundos": round(segundos, 1), "linhas": n_linhas,
                       "faturado": pl.humanizar_bytes(job.total_bytes_billed)})
    print(f"ok  {segundos:5.1f}s  {n_linhas:>9,} linhas")

if resultados:
    print(f"\n{DIA_DEMO}: enriquecimento, features e deteccao em "
          f"{sum(r['segundos'] for r in resultados):.1f}s")
    display(pd.DataFrame(resultados).set_index("n"))
else:
    print("camada 2 desligada")

In [ ]:
# As anomalias que o modelo encontrou nesse dia, produzidas ha segundos.
if EXECUTAR_DEMO:
    display(cliente.query(f"""
    SELECT transaction_hash,
           ROUND(normalized_distance, 2) AS distancia,
           centroid_id
    FROM `{DATASET_DEMO}.anomaly_scores`
    WHERE DATE(block_timestamp) = DATE '{DIA_DEMO}' AND is_anomaly
    ORDER BY normalized_distance DESC
    LIMIT 10
    """, job_config=config_job()).result().to_dataframe())

## 6. Camada 3 — Apresentar

Os resultados sobre o ano completo, lidos das tabelas já materializadas. Todas as
queries desta seção são de leitura e rodam em segundos.

### 6.1 Escolha de K

In [ ]:
selecao = consultar(pl.por_numero(etapas, 31))

if selecao is not None:
    display(selecao)

    # Davies-Bouldin: menor e melhor. Uma so medida no eixo -- a distancia
    # quadratica fica na tabela, porque duas escalas diferentes num grafico so
    # nao seriam comparaveis.
    d = selecao.sort_values("davies_bouldin", ascending=False).reset_index(drop=True)
    PRODUCAO = "10%"
    cores = [P["serie"][1] if a == PRODUCAO else P["serie"][0] for a in d["amostra"]]

    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.barh(d["modelo"], d["davies_bouldin"], height=0.6, color=cores)
    for y, v in enumerate(d["davies_bouldin"]):
        ax.text(v, y, f"  {v:.3f}", va="center", ha="left",
                fontsize=9, color=P["tinta_secundaria"])

    ax.set_xlabel("indice Davies-Bouldin (menor e melhor)")
    ax.set_title("Selecao do modelo", loc="left")
    ax.set_xlim(0, d["davies_bouldin"].max() * 1.22)
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)
    ax.legend(handles=[Patch(facecolor=P["serie"][1], label="modelo de producao"),
                       Patch(facecolor=P["serie"][0], label="demais")],
              loc="lower right", bbox_to_anchor=(1, 1.01), ncol=2)
    plt.tight_layout()
    plt.show()

O resultado contraintuitivo do trabalho está aqui: treinar com a **base completa
não** produziu a melhor clusterização. O K-Means é sensível à inicialização, e mais
volume de treino não melhora monotonicamente o agrupamento. Por isso o modelo de
produção é o treinado sobre 10%, e não o de 100%.

### 6.2 O que cada cluster é

In [ ]:
centroides = consultar(pl.por_numero(etapas, 33))

if centroides is not None:
    matriz = centroides.pivot(index="feature", columns="centroid_id", values="valor")
    lim = float(matriz.abs().max().max())

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matriz.values, cmap=pl.mapa_divergente(), aspect="auto",
                   vmin=-lim, vmax=lim)
    ax.set_xticks(range(matriz.shape[1]))
    ax.set_xticklabels([f"c{c}" for c in matriz.columns])
    ax.set_yticks(range(matriz.shape[0]))
    ax.set_yticklabels(matriz.index, fontsize=8)
    ax.set_title("Centroides por feature", loc="left")
    ax.grid(visible=False)

    cb = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
    cb.set_label("valor padronizado", fontsize=9, color=P["tinta_secundaria"])
    cb.outline.set_visible(False)
    plt.tight_layout()
    plt.show()

As features entram no modelo padronizadas, então **zero é a média** e a escala é
divergente: azul abaixo da média, vermelho acima, cinza neutro no meio. Cada coluna
é o retrato de um comportamento — é aqui que os clusters ganham interpretação.

### 6.3 O modelo encontra o que já sabemos ser incomum

In [ ]:
padroes = consultar(pl.por_numero(etapas, 37))

if padroes is not None:
    display(padroes)

    base = padroes[padroes["padrao"].str.startswith("Baseline")]
    alvos = padroes[~padroes["padrao"].str.startswith("Baseline")].sort_values("pct")
    taxa_base = float(base["pct"].iloc[0])

    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.barh(alvos["padrao"], alvos["pct"], height=0.6, color=P["serie"][0])
    for y, (v, n) in enumerate(zip(alvos["pct"], alvos["detectadas"])):
        ax.text(v, y, f"  {v:.1f}%  ({n:,.0f})", va="center", ha="left",
                fontsize=9, color=P["tinta_secundaria"])

    ax.axvline(taxa_base, color=P["critico"], linewidth=2, linestyle="--")
    ax.text(taxa_base, len(alvos) - 0.35, f" baseline {taxa_base:.1f}%",
            color=P["critico"], fontsize=9, va="top")

    ax.set_xlabel("% do padrao sinalizado como anomalia")
    ax.set_title("Deteccao contra heuristicas independentes", loc="left")
    ax.set_xlim(0, max(alvos["pct"].max(), taxa_base) * 1.35)
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

**Esta é a evidência central do trabalho.** O modelo é não supervisionado: nunca
viu um rótulo, e nenhuma dessas heurísticas foi dada a ele. Mesmo assim sinaliza
transações que as regras estruturais reconhecem como atípicas a taxas muito acima
da linha de base de 1% de todas as transações.

As duas linhas metodologicamente mais limpas são **moeda dormente** e **repetição
de valores**, porque as outras compartilham grandezas com features do modelo — a
contagem de inputs e de outputs entra na matriz, então concordância ali é menos
independente do que parece.

### 6.4 Onde cortar

In [ ]:
dist = consultar(pl.por_numero(etapas, 36))

if dist is not None:
    display(dist)

    r = dist.iloc[0]
    quantis = ["p50", "p90", "p95", "p99", "p999"]
    valores = [float(r[q]) for q in quantis]
    cortes = [(">= 5", r["acima_5"]), (">= 10", r["acima_10"]), (">= 20", r["acima_20"])]

    # Duas medidas de escalas diferentes -> dois eixos separados, nunca dois y.
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))

    a1.plot(quantis, valores, marker="o", color=P["serie"][0])
    for x, v in zip(quantis, valores):
        a1.annotate(f"{v:.2f}", (x, v), textcoords="offset points", xytext=(0, 9),
                    ha="center", fontsize=9, color=P["tinta_secundaria"])
    a1.set_ylabel("distancia normalizada")
    a1.set_title("Quantis da distancia", loc="left")
    a1.set_ylim(0, max(valores) * 1.3)

    # Pontos, nao barras: barra mede magnitude por comprimento e exige linha de
    # base zero, que um eixo logaritmico nao tem.
    contagens = [float(c[1]) for c in cortes]
    a2.scatter(contagens, [c[0] for c in cortes], s=110,
               color=P["serie"][0], zorder=3, clip_on=False)
    for y, v in enumerate(contagens):
        a2.annotate(f"{int(v):,}", (v, y), textcoords="offset points",
                    xytext=(0, 13), ha="center", fontsize=9,
                    color=P["tinta_secundaria"])
    a2.set_xscale("log")
    a2.set_xlim(min(contagens) / 4, max(contagens) * 4)
    a2.set_ylim(-0.6, len(cortes) - 0.4)
    a2.set_xlabel("transacoes acima do corte (escala log)")
    a2.set_title("Sensibilidade ao limiar", loc="left")
    a2.grid(axis="y", visible=False)
    a2.set_axisbelow(True)

    plt.tight_layout()
    plt.show()

O `contamination = 0.01` de `ML.DETECT_ANOMALIES` é um **percentil de corte, não
uma estimativa do modelo**: ele define por construção que 1% das transações será
sinalizado. Por isso `gold.anomaly_scores` preserva a `normalized_distance` — com
ela dá para reavaliar qualquer outro limiar sem reprocessar nada, que é exatamente
o que o painel da direita mostra.

### 6.5 Do score para a fila de trabalho

In [ ]:
prioridade = consultar(pl.por_numero(etapas, 41))

if prioridade is not None:
    display(prioridade)

    # Escala ordinal: uma so matiz, do claro ao escuro conforme a severidade.
    RAMPA = {"normal": "#86b6ef", "baixa": "#5598e7",
             "media": "#2a78d6", "alta": "#1c5cab"}
    ordem = ["normal", "baixa", "media", "alta"]
    d = prioridade.set_index("prioridade").reindex(
        [o for o in ordem if o in set(prioridade["prioridade"])]).reset_index()

    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.scatter(d["transacoes"].astype(float), d["prioridade"], s=130, zorder=3,
               color=[RAMPA[p] for p in d["prioridade"]], clip_on=False)
    for y, (v, pct) in enumerate(zip(d["transacoes"].astype(float), d["pct"])):
        ax.annotate(f"{int(v):,}  ({pct:.2f}%)", (v, y),
                    textcoords="offset points", xytext=(0, 14), ha="center",
                    fontsize=9, color=P["tinta_secundaria"])

    ax.set_xscale("log")
    ax.set_xlim(d["transacoes"].astype(float).min() / 5,
                d["transacoes"].astype(float).max() * 5)
    ax.set_ylim(-0.6, len(d) - 0.35)
    ax.set_xlabel("transacoes na faixa (escala log)")
    ax.set_title("Priorizacao operacional", loc="left")
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

A view `gold.v_anomaly_priority` traduz uma distância contínua em faixas de
trabalho. Um score de 2,7 não diz a ninguém o que fazer; "prioridade média,
faixa com N transações no ano" diz. É a diferença entre um modelo e um produto.

### 6.6 As maiores anomalias do ano

In [ ]:
topo = consultar(pl.por_numero(etapas, 38))

if topo is not None:
    display(topo)

## 7. Roteiro de cinco minutos

| Tempo | Seção | Ao vivo |
|---|---|---|
| 0:00–0:40 | Arquitetura: medalhão, o desvio pelo GCS, 112,5 milhões de transações (§2) | — |
| 0:40–1:10 | Evidência: o Gantt do pipeline real (§3) | ~3 s |
| 1:10–1:50 | Verificação: dry run das 27 etapas de escrita (§4) | ~25 s |
| 1:50–2:50 | Demonstração: enriquecimento, features e detecção sobre um dia (§5) | ~15 s |
| 2:50–3:20 | Escolha de K e o resultado contraintuitivo (§6.1) | ~3 s |
| 3:20–4:10 | Detecção contra heurísticas independentes (§6.3) | ~4 s |
| 4:10–4:40 | Priorização operacional (§6.5) | ~3 s |
| 4:40–5:00 | Maiores anomalias e fechamento (§6.6) | ~2 s |

**§6.2 (centroides) e §6.4 (limiar) são as duas seções cortáveis.** Elas aprofundam
a interpretação, mas o argumento se sustenta sem elas — deixe-as prontas para o
caso de a banca perguntar.

## 8. Limpeza

O dataset `demo` é descartável e custa armazenamento enquanto existir. Como a conta
é emprestada, vale remover ao final.

In [ ]:
# Descomente para remover as saidas da demonstracao.
# cliente.delete_dataset(f"{cfg['PROJECT_ID']}.{DATASET_DEMO}",
#                        delete_contents=True, not_found_ok=True)
# print("dataset de demonstracao removido")